# Asistente Fiscal con Gemini, RAG y LangGraph

Agente experto para gestorías españolas. Asesora sobre obligaciones fiscales de **autónomos y sociedades**: cómo rellenar declaraciones, plazos del calendario fiscal y avisos de antelación.

**Stack:** Google Gemini · ChromaDB · LangGraph · LangChain

---

## Índice

1. [Instalación y configuración](#1-instalación-y-configuración)
2. [Carga y procesado de documentos](#2-carga-y-procesado-de-documentos)
3. [Creación de la base de conocimiento vectorial](#3-creación-de-la-base-de-conocimiento-vectorial)
4. [Diseño del agente LangGraph](#4-diseño-del-agente-langgraph)
   - 4a. System prompt con few-shot examples
   - 4b. Grafo con routing condicional y gestión de tokens
5. [Lógica de avisos por antelación](#5-lógica-de-avisos-por-antelación)
6. [Moderación en cascada](#6-moderación-en-cascada)
7. [Herramientas fiscales con @tool y loop ReAct](#7-herramientas-fiscales-con-tool-y-loop-react)
8. [LLM-as-Judge — evaluación de calidad](#8-llm-as-judge--evaluación-de-calidad)
9. [Demo interactiva](#9-demo-interactiva)

---

### Arquitectura del graph


```
START
  │
  ▼
podar_historial    ← elimina mensajes si historial > MAX_MESSAGES
  │
  ▼
detectar_perfil    ← infiere 'autonomo' o 'sociedad' del historial (SRP)
  │
  ▼
clasificar_consulta  ← detecta tipo: plazos | documentos | general
  │
  ├─► recuperar_plazos     (prioriza CSVs de calendario)
  ├─► recuperar_documentos (prioriza manuales PDF)
  └─► recuperar_general    (mezcla balanceada)
         │
         ▼
    generar_respuesta  ← Gemini + RAG context + historial
         │
         ▼
        END
```

## 1. Instalación y configuración

In [12]:
# %pip install langchain langchain-google-genai langchain-community langchain-chroma langchain-text-splitters langchain-experimental langgraph chromadb pypdf pdfplumber python-dotenv pandas scikit-learn numpy sentence-transformers streamlit

  Obtaining dependency information for langchain-chroma from https://files.pythonhosted.org/packages/ae/35/2a6d1191acaad043647e28313b0ecd161d61f09d8be37d1996a90d752c13/langchain_chroma-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for streamlit from https://files.pythonhosted.org/packages/d8/1a/3ca2293d8552bacea3e67e9600d2d1df7df4a325059769ad83d91c279595/streamlit-1.57.0-py3-none-any.whl.metadata
  Obtaining dependency information for altair!=5.4.0,!=5.4.1,<7,>=4.0 from https://files.pythonhosted.org/packages/ce/63/5dacc8d8306c715088b897a479e551bc0779fd2f0f26c97fec5e36542b4e/altair-6.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for blinker<2,>=1.5.0 from https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for cachetools<8,>=5.5 from https://files.pythonhosted.org/packages/bf/0f/f897abe4ea0a8c408ae65c8c83bffab4936ad65d

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# Carga automática: GOOGLE_API_KEY, GOOGLE_API_KEY_2, GOOGLE_API_KEY_3, ...
# Añadir más claves al .env no requiere cambios en el código.
_base = os.getenv("GOOGLE_API_KEY")
GOOGLE_API_KEYS = [_base] if _base else []
i = 2
while True:
    k = os.getenv(f"GOOGLE_API_KEY_{i}")
    if not k:
        break
    GOOGLE_API_KEYS.append(k)
    i += 1

assert GOOGLE_API_KEYS, "No hay ninguna GOOGLE_API_KEY configurada en el archivo .env"
print(f"Claves API cargadas: {len(GOOGLE_API_KEYS)}")

Claves API cargadas: 3


## 2. Carga y procesado de documentos

In [14]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
import pdfplumber

# Rutas organizadas por tipo
PRACTICOS_ES = Path("../data/manuales/practicos/es")
WEB_ES       = Path("../data/manuales/web/es")

MANUAL_METADATA = {
    (PRACTICOS_ES, "manual_iva_303_2025.pdf"):                   {"modelos": "303",     "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_actividades_economicas_111_115.pdf"):  {"modelos": "111,115", "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte1.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte2.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_sociedades_200_202_2024.pdf"):         {"modelos": "200,202", "perfil": "sociedad", "tipo": "manual_practico", "idioma": "es"},
    (WEB_ES, "manual_rentaweb_100_2024.pdf"):                     {"modelos": "100",     "perfil": "autonomo", "tipo": "manual_web",      "idioma": "es"},
    (WEB_ES, "manual_sociedadesweb_200_2024.pdf"):                {"modelos": "200",     "perfil": "sociedad", "tipo": "manual_web",      "idioma": "es"},
}

CHROMA_DIR_CHECK = Path("../chroma_db")

# Los embeddings se inicializan aquí, antes de cargar los PDFs, para que el
# SemanticChunker y el vectorstore (sección 3) compartan la misma instancia
# sin duplicar la carga del modelo ni crear embeddings inconsistentes.
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Modelo de embeddings cargado: {EMBEDDING_MODEL}")

if CHROMA_DIR_CHECK.exists():
    docs_manuales = []
    print("Base de conocimiento ya indexada — carga de PDFs omitida.")
else:
    # SemanticChunker respeta fronteras conceptuales (artículos, apartados) en lugar de
    # cortar por número fijo de caracteres. Crítico para documentos fiscales donde un
    # apartado completo es la unidad mínima de información coherente.
    semantic_splitter = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )
    # Fallback para textos muy cortos donde el chunker semántico no puede actuar
    fallback_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)

    def cargar_pdfs(metadata_map: dict) -> list:
        """Carga PDFs con pdfplumber, divide semánticamente y añade metadatos."""
        docs = []
        for (directorio, filename), meta in metadata_map.items():
            path = directorio / filename
            if not path.exists():
                print(f"  [AVISO] No encontrado: {path}")
                continue
            texto_completo = []
            with pdfplumber.open(str(path)) as pdf:
                for page in pdf.pages:
                    texto = page.extract_text()
                    if texto:
                        texto_completo.append(texto)
            texto_unido = "\n\n".join(texto_completo)
            doc_base = Document(page_content=texto_unido, metadata={**meta, "fuente": filename})
            try:
                chunks = semantic_splitter.split_documents([doc_base])
                if not chunks:
                    raise ValueError("SemanticChunker devolvió 0 chunks")
            except Exception:
                chunks = fallback_splitter.split_documents([doc_base])
            docs.extend(chunks)
            print(f"  [{meta['idioma']}] {filename}: {len(chunks)} chunks (semántico)")
        return docs

    print("Cargando manuales con chunking semántico...")
    docs_manuales = cargar_pdfs(MANUAL_METADATA)
    print(f"\nTotal chunks manuales: {len(docs_manuales)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embeddings cargado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Base de conocimiento ya indexada — carga de PDFs omitida.


In [15]:
import pandas as pd
from langchain_core.documents import Document

CALENDARIO_PATH   = Path("../data/calendario_fiscal.csv")
OBLIGACIONES_PATH = Path("../data/obligaciones_perfil.csv")

if CHROMA_DIR_CHECK.exists():
    docs_calendario   = []
    docs_obligaciones = []
    print("CSVs omitidos — base de conocimiento ya indexada.")
else:
    def cargar_csv_como_docs(path: Path, tipo: str, sep: str = ",") -> list:
        df = pd.read_csv(path, sep=sep)
        docs = []
        for _, row in df.iterrows():
            contenido = " | ".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))
            meta = {"fuente": path.name, "tipo": tipo}
            if "modelo"   in row: meta["modelos"]   = str(row["modelo"])
            if "perfil"   in row: meta["perfil"]    = str(row["perfil"])
            if "trimestre" in row: meta["trimestre"] = str(row["trimestre"])
            docs.append(Document(page_content=contenido, metadata=meta))
        return docs

    docs_calendario   = cargar_csv_como_docs(CALENDARIO_PATH,   "calendario",         sep=",")
    docs_obligaciones = cargar_csv_como_docs(OBLIGACIONES_PATH, "obligaciones_perfil", sep=";")

    print(f"Chunks calendario:   {len(docs_calendario)}")
    print(f"Chunks obligaciones: {len(docs_obligaciones)}")
    print(f"\nTotal a indexar: {len(docs_manuales) + len(docs_calendario) + len(docs_obligaciones)}")


CSVs omitidos — base de conocimiento ya indexada.


## 3. Creación de la base de conocimiento vectorial

In [16]:
import chromadb
from langchain_chroma import Chroma

# 'embeddings' ya está definido en la sección 2 — se reutiliza aquí para
# garantizar que el vectorstore usa exactamente el mismo modelo con el que
# se generaron los chunks, evitando inconsistencias en el espacio vectorial.
CHROMA_DIR = "../chroma_db"
COLLECTION_NAME = "base_fiscal"

if Path(CHROMA_DIR).exists():
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME
    )
    print(f"Base de conocimiento cargada desde disco: {vectorstore._collection.count()} documentos")
else:
    all_docs = docs_manuales + docs_calendario + docs_obligaciones
    print(f"Total documentos a indexar: {len(all_docs)}")
    print("Indexando en local... (puede tardar 5-10 min)")

    vectorstore = Chroma.from_documents(
        documents=all_docs,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME
    )
    print(f"\nBase de conocimiento creada: {vectorstore._collection.count()} documentos")

Base de conocimiento cargada desde disco: 2158 documentos


In [17]:
# Verificar la colección con consultas de prueba
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

test_queries = [
    "¿Cuándo es el plazo del modelo 303 del primer trimestre?",
    "¿Cómo se calcula la base imponible del IVA?",
    "¿Qué obligaciones fiscales tiene un autónomo en el primer trimestre?",
]

for q in test_queries:
    print(f"\nConsulta: {q}")
    results = retriever.invoke(q)
    for r in results:
        print(f"  [{r.metadata.get('fuente', '?')}] {r.page_content[:120]}...")

# Diagnóstico: ver qué idioma tiene el texto extraído de los manuales de renta
print("\n\n--- DIAGNÓSTICO: primeras líneas del manual de renta ---")
if docs_manuales:
    for doc in docs_manuales[:3]:
        if "renta" in doc.metadata.get("fuente", ""):
            print(f"Fuente: {doc.metadata['fuente']}")
            print(f"Texto: {doc.page_content[:300]}")
            print("---")


Consulta: ¿Cuándo es el plazo del modelo 303 del primer trimestre?
  [calendario_fiscal.csv] modelo: 303 | nombre: Autoliquidación IVA 3T 2026 | perfil: ambos | trimestre: 3T_2026 | fecha_limite_2026: 2026-10-20 |...
  [calendario_fiscal.csv] modelo: 303 | nombre: Autoliquidación IVA 1T 2026 | perfil: ambos | trimestre: 1T_2026 | fecha_limite_2026: 2026-04-20 |...
  [manual_renta_100_130_2025_parte1.pdf] Página 1185

Capítulo 18. Cuota líquida, cuota resultante de la autoliquidación, cuota diferencial y resultado de la dec...
  [calendario_fiscal.csv] modelo: 303 | nombre: Autoliquidación IVA 2T 2026 | perfil: ambos | trimestre: 2T_2026 | fecha_limite_2026: 2026-07-20 |...

Consulta: ¿Cómo se calcula la base imponible del IVA?
  [manual_iva_303_2025.pdf] Los autoconsumos de bienes y servicios. Contenido
El impuesto se devengará:
01/12/2025 - Manual práctico IVA 2025. Págin...
  [manual_renta_100_130_2025_parte1.pdf] Página 421

Capítulo 7. Rendimientos de actividades económicas. Métod

## 4. Diseño del agente LangGraph

In [18]:
SYSTEM_PROMPT = """Eres un asesor fiscal experto de una gestoría española llamada GestorIA.
Tu función es ayudar a gestores y clientes con las obligaciones fiscales de autónomos y sociedades en España.

## ROL Y LÍMITES

Eres un asistente especializado EXCLUSIVAMENTE en fiscalidad española. No respondas preguntas fuera de este ámbito.
Si te preguntan algo que no es fiscal (contabilidad general, derecho laboral, etc.), indica amablemente que está fuera de tu alcance.

## IDIOMA

Detecta el idioma en que escribe el usuario y responde siempre en ese mismo idioma.
El idioma de los documentos recuperados (contexto) NO influye en tu idioma de respuesta.

## FUENTES Y JERARQUÍA

Aplica siempre esta prioridad al responder:

1. **Contexto RAG** (documentos recuperados) — máxima autoridad para datos concretos: fechas, casillas, porcentajes, plazos, importes. Si el RAG y tu conocimiento general difieren, prevalece siempre el RAG.
2. **Conocimiento general como asesor fiscal** — solo para procedimientos estándar (acceso a sede AEAT, Cl@ve, certificado digital). Cita como "Procedimiento estándar AEAT".
3. **Ninguna fuente** — si no hay datos en ninguna de las dos fuentes anteriores, declara la ausencia explícitamente.

Si tienes información parcial, responde con lo que tengas y señala qué falta: "Sobre X dispongo de [dato], pero no tengo información sobre Y en mi base de conocimiento."
Solo cierra con "No dispongo de información suficiente..." cuando no tengas absolutamente ningún dato relevante.

## FUENTES — CÓMO CITARLAS

Al citar la fuente usa el nombre descriptivo, no solo el nombre de fichero:
- `calendario_fiscal.csv` → "Calendario fiscal AEAT 2026"
- `obligaciones_perfil.csv` → "Mapa de obligaciones por perfil"
- `manual_iva_303_2025.pdf` → "Manual práctico IVA 303 (AEAT 2025)"
- `manual_renta_100_130_2025_parte1.pdf` / `parte2.pdf` → "Manual práctico Renta 100/130 (AEAT 2025)"
- `manual_sociedades_200_202_2024.pdf` → "Manual práctico Sociedades 200/202 (AEAT 2024)"
- `manual_actividades_economicas_111_115.pdf` → "Manual Actividades Económicas 111/115 (AEAT)"
- `manual_rentaweb_100_2024.pdf` → "Manual RentaWeb 100 (AEAT 2024)"
- `manual_sociedadesweb_200_2024.pdf` → "Manual SociedadesWeb 200 (AEAT 2024)"
- Conocimiento propio de procedimiento → "Procedimiento estándar AEAT"

## VIGENCIA DEL CONTEXTO

Si en los fragmentos RAG recuperados detectas referencias a ejercicios anteriores al trimestre actual (por ejemplo, menciones a "2023", "2024" o años anteriores en fechas de plazo o nombres de modelos), añade al final de tu respuesta: "⚠️ Parte del contexto recuperado puede corresponder a ejercicios anteriores. Verifica los datos en la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) antes de actuar."
No añadas este aviso si los fragmentos son coherentes con el ejercicio fiscal actual (2025–2026).

## IDENTIFICACIÓN DE PERFIL

- Identifica el perfil del cliente antes de responder: autónomo, sociedad, o ambos.
- Si el perfil aparece en la línea "Perfil del cliente" al inicio del mensaje, úsalo directamente sin volver a preguntar.
- Si el perfil NO está claro ni en esa línea ni en el historial, PREGUNTA antes de responder. No asumas.
- Una vez identificado, el perfil persiste durante toda la conversación. Inclúyelo en la primera respuesta y en aquellas donde aporte claridad (cambio de modelo, respuesta larga). En respuestas de seguimiento cortas puede omitirse si ya es evidente del contexto.
- En preguntas de seguimiento ("¿y el 130?", "¿cuánto tengo que pagar?"), usa el perfil y contexto del turno anterior sin solicitar aclaración si la pregunta es razonablemente interpretable.
- Si el modelo preguntado no aplica al perfil del cliente, indícalo antes de responder y redirige al modelo correcto cuando sea posible. Ejemplos: "El modelo 130 no aplica a sociedades — el equivalente es el modelo 202." / "El modelo 200 es exclusivo de sociedades — los autónomos liquidan el IRPF con el modelo 100."

## NIVEL TÉCNICO

Adapta el nivel de detalle según quién pregunta:
- **Gestor / asesor fiscal** — usa terminología técnica (casillas, regímenes, base imponible, devengo). Respuestas densas y precisas.
- **Cliente final** — lenguaje claro, sin jerga. Explica brevemente qué significa cada término técnico la primera vez que lo uses.

Si no está claro el tipo de interlocutor, usa un nivel intermedio: terminología técnica con una frase de contexto cuando sea necesario.

## ESTRUCTURA DE RESPUESTA

Adapta la estructura al tipo de pregunta. Incluye SOLO los apartados relevantes:

**Preguntas de plazo o calendario** → Perfil | Modelo + fecha límite + domiciliación + inicio preparación | Fuente
**Preguntas de cumplimentación o casillas** → Perfil | Nombre de la casilla + explicación técnica | Fuente
**Preguntas de procedimiento o pasos** → Perfil | Lista numerada de pasos | Fuente
**Preguntas de obligaciones generales** → Perfil | Lista de modelos aplicables con plazo e inicio preparación | Fuente
**Preguntas mixtas** → combina las secciones necesarias en orden lógico, sin duplicar información.

Nunca incluyas secciones vacías ni encabezados sin contenido. Si la pregunta solo pide un dato concreto (una fecha, una casilla), responde directamente.

## PLAZOS Y ANTELACIÓN

- La fecha de hoy y el trimestre activo aparecen en la línea "Fecha de hoy — Trimestre actual" del mensaje. Úsalos para resolver preguntas sin trimestre explícito ("¿qué tengo pendiente?", "¿el trimestre que viene?"). Si esa línea no está presente, deduce el trimestre a partir de tu conocimiento de la fecha actual: enero–marzo=1T, abril–junio=2T, julio–septiembre=3T, octubre–diciembre=4T.
- Días de preparación recomendados por tipo de obligación (úsalos si el contexto RAG no especifica otro valor):
  - Modelos trimestrales (303, 130, 111, 115, 202): 10 días antes del plazo
  - Modelos anuales simples (390, 347): 15 días antes del plazo
  - Modelos anuales complejos (100, 200): 30 días antes del plazo
- Cuando informes de un plazo, calcula y muestra siempre la fecha de inicio de preparación.
- **DOMICILIACIÓN — REGLA OBLIGATORIA:** Si el contexto RAG incluye el campo `domiciliacion_hasta` para el modelo consultado, muéstralo SIEMPRE en la respuesta con el formato "Domiciliación hasta: [fecha]", inmediatamente después de la fecha límite. La domiciliación bancaria adelanta el plazo efectivo de pago y es información crítica para el cliente. Nunca la omitas si está disponible en el contexto.
- Si el usuario pregunta "¿qué tengo pendiente?", lista TODAS las obligaciones del trimestre activo ordenadas por fecha límite.

## CORRECCIÓN DE ERRORES DEL USUARIO

Si detectas una incoherencia en la pregunta (trimestre incorrecto para ese modelo, fecha imposible, modelo que no aplica al perfil), corrígela de forma breve y directa antes de responder:
"El modelo 130 no tiene presentación en el cuarto trimestre — el último es en octubre (3T). Te respondo sobre el 3T:"

## CONTEXTO ACUMULADO EN CONVERSACIÓN

Si el usuario hace varias preguntas encadenadas sobre el mismo modelo o tema, no repitas explicaciones ya dadas en el mismo hilo. Céntrate solo en la información nueva que aporta la pregunta actual. Si necesitas referirte a algo ya explicado, usa una referencia breve: "Como comenté antes, el plazo es el 20 de julio."

## TONO

Profesional y directo. Usa listas y negritas para facilitar la lectura.
Evita relleno vacío ("¡Claro!", "¡Por supuesto!") pero puedes usar una transición breve cuando el contexto lo pida ("En ese caso," "Para este perfil,").

---

## EJEMPLOS DE RESPUESTA CORRECTA

**Ejemplo 1 — Plazo de un modelo concreto:**
Usuario: "Soy autónomo, ¿cuándo presento el 303 del 2T?"

Respuesta:
**Perfil:** Autónomo.
**Modelo 303 — Autoliquidación IVA 2T 2026:**
- Fecha límite: 20 de julio de 2026
- Domiciliación hasta: 15 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026 (10 días antes)
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 2 — Perfil no especificado:**
Usuario: "¿Cuándo tengo que presentar el modelo 303?"

Respuesta:
Para darte la información correcta, necesito saber tu perfil fiscal. ¿Eres autónomo o representas a una sociedad?

---

**Ejemplo 3 — Cómo rellenar una casilla:**
Usuario: "Soy autónomo. ¿Cómo relleno la casilla 01 del modelo 303?"

Respuesta:
**Perfil:** Autónomo.
**Casilla 01 — Base imponible al tipo general (21%):**
Incluye el importe total de las entregas de bienes y prestaciones de servicios sujetas y no exentas de IVA gravadas al 21%, sin incluir el propio impuesto.
*Fuente: Manual práctico IVA 303 (AEAT 2025)*

---

**Ejemplo 4 — Obligaciones de un trimestre:**
Usuario: "Somos una S.L. ¿Qué tenemos que presentar en el segundo trimestre?"

Respuesta:
**Perfil:** Sociedad.
**Obligaciones 2T 2026:**
- Modelo 303 — IVA 2T | Plazo: 20 julio | Domiciliación hasta: 15 julio | Inicio preparación: 10 julio
- Modelo 111 — Retenciones IRPF 2T | Plazo: 20 julio | Domiciliación hasta: 15 julio | Inicio preparación: 10 julio
- Modelo 115 — Retenciones alquileres 2T | Plazo: 20 julio | Domiciliación hasta: 15 julio | Inicio preparación: 10 julio
- Modelo 202 — Pago fraccionado IS (si cuota íntegra último IS > 0) | Plazo: 20 julio | Domiciliación hasta: 15 julio | Inicio preparación: 10 julio
*Fuente: Calendario fiscal AEAT 2026, Mapa de obligaciones por perfil*

---

**Ejemplo 5 — Información no disponible:**
Usuario: "¿Cuál es el tipo de recargo de equivalencia para el aceite de oliva?"

Respuesta:
No dispongo de información suficiente sobre este punto en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) o al gestor responsable.

---

**Ejemplo 6 — Pasos para presentar un modelo:**
Usuario: "Represento una sociedad. ¿Qué pasos debo seguir para presentar el IVA?"

Respuesta:
**Perfil:** Sociedad.
**Pasos para presentar el Modelo 303 — Autoliquidación IVA:**

1. **Recopilar la documentación** — Facturas emitidas (IVA repercutido) y facturas recibidas (IVA soportado) del trimestre.
2. **Calcular el resultado** — IVA repercutido − IVA soportado deducible. Si es positivo, hay que ingresar; si es negativo, se puede compensar o solicitar devolución.
3. **Acceder a la Sede Electrónica de la AEAT** — Con certificado digital, DNIe o Cl@ve PIN en sede.agenciatributaria.gob.es.
4. **Cumplimentar el formulario** — Ir a "Trámites destacados > Modelo 303". Rellenar casillas 01-09 (IVA devengado) y 28-44 (IVA deducible).
5. **Presentar y pagar si procede** — Se puede domiciliar hasta 5 días antes del plazo límite.
6. **Guardar el justificante** — La AEAT emite un CSV de confirmación como acuse de recibo oficial.

*Fuente: Manual práctico IVA 303 (AEAT 2025), procedimiento estándar AEAT*

---

**Ejemplo 7 — Pregunta de seguimiento (perfil ya conocido):**
[Turno anterior: el usuario dijo ser autónomo y preguntó por el modelo 303]
Usuario: "¿Y el 130?"

Respuesta:
**Modelo 130 — Pago fraccionado IRPF 2T 2026** (Autónomo, estimación directa):
- Fecha límite: 20 de julio de 2026
- Domiciliación hasta: 15 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 8 — Información parcialmente disponible:**
Usuario: "Soy autónomo. ¿Cuánto tengo que ingresar en el modelo 130 y cuál es la penalización por presentarlo tarde?"

Respuesta:
**Perfil:** Autónomo.
**Modelo 130 — Cálculo del importe:** El importe es el 20% del rendimiento neto de actividades económicas del trimestre, menos retenciones soportadas y pagos fraccionados anteriores del ejercicio.
*Fuente: Manual práctico Renta 100/130 (AEAT 2025)*

Sobre las penalizaciones por presentación fuera de plazo no dispongo de información en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) para conocer el régimen de recargos aplicable.

---

**Ejemplo 9 — Corrección de error del usuario:**
Usuario: "Soy autónomo, ¿cuándo presento el modelo 130 del cuarto trimestre?"

Respuesta:
El modelo 130 no tiene presentación en el cuarto trimestre — el último pago fraccionado del año es el 3T (octubre). En enero se presenta el modelo 100 con la liquidación anual del IRPF.

**Modelo 130 — Pago fraccionado IRPF 3T 2026** (Autónomo):
- Fecha límite: 20 de octubre de 2026
- Domiciliación hasta: 15 de octubre de 2026
- Inicio de preparación recomendado: 10 de octubre de 2026
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 10 — Pregunta mixta (plazo + procedimiento):**
Usuario: "Soy autónomo. ¿Cuándo tengo que presentar el 303 del 1T y cómo lo hago?"

Respuesta:
**Perfil:** Autónomo.

**Plazo — Modelo 303 IVA 1T 2026:**
- Fecha límite: 20 de abril de 2026
- Domiciliación hasta: 15 de abril de 2026
- Inicio de preparación recomendado: 10 de abril de 2026
*Fuente: Calendario fiscal AEAT 2026*

**Cómo presentarlo:**
1. Accede a sede.agenciatributaria.gob.es con certificado digital o Cl@ve PIN.
2. Ve a "Trámites destacados > Modelo 303" y cumplimenta las casillas con los datos del trimestre.
3. Presenta y, si sale a ingresar, puedes domiciliar el pago hasta el 15 de abril.
4. Guarda el justificante con el CSV de confirmación.
*Fuente: procedimiento estándar AEAT*

---

**Ejemplo 11 — Modelo no aplicable al perfil:**
Usuario: "Soy autónomo, ¿cuándo presento el modelo 202?"

Respuesta:
El modelo 202 (Pagos fraccionados del Impuesto sobre Sociedades) no aplica a autónomos — es exclusivo de sociedades. Como autónomo, el pago fraccionado del IRPF se realiza con el **modelo 130** (estimación directa) o el **modelo 131** (estimación objetiva/módulos).

**Modelo 130 — Pago fraccionado IRPF 2T 2026** (Autónomo):
- Fecha límite: 20 de julio de 2026
- Domiciliación hasta: 15 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026
*Fuente: Calendario fiscal AEAT 2026, Mapa de obligaciones por perfil*
"""

print("System prompt configurado con 11 few-shot examples.")

System prompt configurado con 11 few-shot examples.


In [19]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage
import operator
import re
import datetime
import time

# ── Constantes ──────────────────────────────────────────────────────────────
MAX_MESSAGES    = 10
MAX_RETRIES_RPM = 5

# Matriz de fallback: agente usa modelos potentes primero; lite usa modelos ligeros primero.
MODELOS_AGENTE = [
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-3.1-flash-lite-preview",
]
MODELOS_LITE = [
    "gemini-3.1-flash-lite-preview",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3-flash-preview",
]

class AgentState(TypedDict):
    messages:      Annotated[list, operator.add]
    perfil:        str
    contexto_rag:  str
    tipo_consulta: str


def _extraer_retry_delay(error_str: str, default: float = 15.0) -> float:
    match = re.search(r"retryDelay.*?(\d+(?:\.\d+)?)\s*s", error_str)
    return float(match.group(1)) + 1 if match else default

def _es_limite_diario(err: str) -> bool:
    return "GenerateRequestsPerDayPerProjectPerModel" in err

def _es_limite_rpm(err: str) -> bool:
    return "GenerateRequestsPerMinutePerProjectPerModel" in err

def _crear_llm(modelo: str, clave_idx: int) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(
        model=modelo,
        google_api_key=GOOGLE_API_KEYS[clave_idx],
        temperature=0,
    )

# Estado global: índice de modelo y clave activos para cada tipo
_estado = {
    "agente": {"modelo_idx": 0, "clave_idx": 0},
    "lite":   {"modelo_idx": 0, "clave_idx": 0},
}

def _siguiente_combinacion(tipo: str) -> bool:
    """Avanza a la siguiente combinación modelo+clave. Devuelve False si se agotaron todas."""
    modelos = MODELOS_AGENTE if tipo == "agente" else MODELOS_LITE
    est = _estado[tipo]
    if est["clave_idx"] + 1 < len(GOOGLE_API_KEYS):
        est["clave_idx"] += 1
        return True
    if est["modelo_idx"] + 1 < len(modelos):
        est["modelo_idx"] += 1
        est["clave_idx"] = 0
        return True
    return False

def _llm_actual(tipo: str) -> ChatGoogleGenerativeAI:
    modelos = MODELOS_AGENTE if tipo == "agente" else MODELOS_LITE
    est = _estado[tipo]
    return _crear_llm(modelos[est["modelo_idx"]], est["clave_idx"])

def _invoke_con_retry(llm_obj, messages: list, tipo: str = "agente"):
    """Invoca el LLM con retry RPM y fallback modelo+clave ante límite diario.
    Devuelve solo la respuesta. Si hubo fallback, actualiza el LLM global del tipo."""
    modelos = MODELOS_AGENTE if tipo == "agente" else MODELOS_LITE
    for _ in range(MAX_RETRIES_RPM + len(GOOGLE_API_KEYS) * len(modelos)):
        try:
            return llm_obj.invoke(messages)
        except Exception as e:
            err = str(e)
            if "RESOURCE_EXHAUSTED" not in err and "429" not in err:
                raise
            if _es_limite_rpm(err):
                delay = _extraer_retry_delay(err)
                print(f"[WARN] Límite RPM — esperando {delay:.0f}s...")
                time.sleep(delay)
                continue
            if _es_limite_diario(err):
                modelo_actual = modelos[_estado[tipo]["modelo_idx"]]
                if not _siguiente_combinacion(tipo):
                    raise RuntimeError("Todas las combinaciones modelo+clave están agotadas.") from e
                nuevo_modelo = modelos[_estado[tipo]["modelo_idx"]]
                nueva_clave  = _estado[tipo]["clave_idx"] + 1
                print(f"[WARN] [{modelo_actual}] agotado → {nuevo_modelo} clave {nueva_clave}")
                llm_obj = _llm_actual(tipo)
                # Actualiza el LLM global para que los nodos siguientes usen el nuevo
                _llms_globales[tipo] = llm_obj
                continue
            raise
    raise RuntimeError("Se agotaron los reintentos de la API.")

# Registro de LLMs activos — evita variables globales sueltas modificadas por funciones
_llms_globales: dict = {}

def _inicializar_llm(tipo: str = "agente") -> ChatGoogleGenerativeAI:
    """Busca la primera combinación modelo+clave disponible al arrancar."""
    modelos = MODELOS_AGENTE if tipo == "agente" else MODELOS_LITE
    if not GOOGLE_API_KEYS:
        raise RuntimeError("No hay ninguna GOOGLE_API_KEY configurada.")
    for m_idx, modelo in enumerate(modelos):
        for c_idx in range(len(GOOGLE_API_KEYS)):
            try:
                llm_test = _crear_llm(modelo, c_idx)
                llm_test.invoke([HumanMessage(content="1+1=")])
                _estado[tipo]["modelo_idx"] = m_idx
                _estado[tipo]["clave_idx"]  = c_idx
                if m_idx > 0 or c_idx > 0:
                    print(f"[INFO] [{tipo}] Arrancando con {modelo} / clave {c_idx + 1}")
                return llm_test
            except Exception as e:
                err = str(e)
                if "RESOURCE_EXHAUSTED" not in err and "429" not in err:
                    raise
                if _es_limite_diario(err):
                    print(f"[WARN] [{tipo}] {modelo} clave {c_idx + 1} agotada (límite diario)...")
                    continue
                time.sleep(_extraer_retry_delay(err))
    raise RuntimeError(f"Todas las combinaciones modelo+clave están agotadas al arrancar ({tipo}).")


llm      = _inicializar_llm("agente")
llm_lite = _inicializar_llm("lite")
_llms_globales["agente"] = llm
_llms_globales["lite"]   = llm_lite

# ── Patrones para detectar el perfil del cliente ────────────────────────────
_KW_AUTO = re.compile(r"\baut[oó]nomo\b", re.IGNORECASE)
_KW_SOC  = re.compile(r"\b(sociedad|empresa|s\.l|s\.a)\b", re.IGNORECASE)

# ── Gestión de tokens: poda de historial ────────────────────────────────────
def podar_historial(state: AgentState) -> AgentState:
    mensajes = state["messages"]
    if len(mensajes) <= MAX_MESSAGES:
        return {}
    n_eliminar = len(mensajes) - MAX_MESSAGES
    if n_eliminar % 2 != 0:
        n_eliminar += 1
    n_eliminar = min(n_eliminar, len(mensajes))
    ids_a_eliminar = [RemoveMessage(id=m.id) for m in mensajes[:n_eliminar]]
    print(f"[Historial] Podados {n_eliminar} mensajes antiguos. Quedan {len(mensajes) - n_eliminar}.")
    return {"messages": ids_a_eliminar}

# ── Nodo: detecta el perfil del cliente a partir del historial ───────────────
def detectar_perfil(state: AgentState) -> AgentState:
    """Responsabilidad única: inferir 'autonomo' o 'sociedad' del historial.
    Se ejecuta antes de recuperar contexto para que los retrievers puedan filtrar por perfil."""
    if state.get("perfil"):
        return {}
    for msg in reversed(state["messages"]):
        texto = msg.content if isinstance(msg.content, str) else ""
        if _KW_AUTO.search(texto):
            print("[Perfil] Detectado: autónomo")
            return {"perfil": "autonomo"}
        if _KW_SOC.search(texto):
            print("[Perfil] Detectado: sociedad")
            return {"perfil": "sociedad"}
    return {}

# ── Clasificador: detecta tipo de consulta ──────────────────────────────────
_KEYWORDS_PLAZOS = re.compile(
    r"\b(plazo|fecha|cuando|cuándo|vencimiento|trimestre|domicili|antelacion|antelación|pendiente)\b",
    re.IGNORECASE
)
_KEYWORDS_DOCS = re.compile(
    r"\b(casilla|rellenar|cumplimentar|calcul|base imponible|deduccion|deducción|como se|cómo se|"
    r"instruccion|instrucción|apartado|anexo|paso|pasos|proceso|procedimiento|"
    r"c[oó]mo presento|c[oó]mo se presenta|c[oó]mo funciona|c[oó]mo hago|"
    r"c[oó]mo debo|qué pasos|qu[eé] debo hacer)\b",
    re.IGNORECASE
)

def clasificar_consulta(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    if _KEYWORDS_DOCS.search(ultima):
        tipo = "documentos"
    elif _KEYWORDS_PLAZOS.search(ultima):
        tipo = "plazos"
    else:
        tipo = "general"
    print(f"[Router] Consulta clasificada como: '{tipo}'")
    return {"tipo_consulta": tipo}

def router(state: AgentState) -> Literal["recuperar_plazos", "recuperar_documentos", "recuperar_general"]:
    return {
        "plazos":     "recuperar_plazos",
        "documentos": "recuperar_documentos",
        "general":    "recuperar_general",
    }[state["tipo_consulta"]]

# ── Nodos de recuperación RAG ────────────────────────────────────────────────
def _combinar_docs(docs_a: list, docs_b: list) -> str:
    vistos = set()
    combinados = []
    for doc in docs_a + docs_b:
        clave = doc.page_content[:100]
        if clave not in vistos:
            vistos.add(clave)
            combinados.append(doc)
    resultado = "\n\n".join(
        f"[{d.metadata.get('fuente', '?')}]\n{d.page_content}"
        for d in combinados
    )
    print(f"[RAG] Documentos recuperados: {len(combinados)} "
          f"({', '.join(set(d.metadata.get('fuente','?') for d in combinados))})")
    return resultado

def recuperar_plazos(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 10, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    search_manuales = {"k": 3}
    if perfil in ("autonomo", "sociedad"):
        search_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_manuales).invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_csv, docs_manuales_res)}

def recuperar_documentos(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    search_manuales = {"k": 12}
    if perfil in ("autonomo", "sociedad"):
        search_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_manuales).invoke(ultima)
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_manuales_res, docs_csv)}

def recuperar_general(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    search_kwargs_manuales = {"k": 5}
    if perfil in ("autonomo", "sociedad"):
        search_kwargs_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_kwargs_manuales).invoke(ultima)
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 6, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_manuales_res, docs_csv)}

# ── Nodo de generación ───────────────────────────────────────────────────────
def generar_respuesta(state: AgentState) -> AgentState:
    """Responsabilidad única: construir el prompt con el contexto RAG e invocar el LLM."""
    contexto  = state.get("contexto_rag", "")
    historial = state["messages"]

    hoy = datetime.date.today()
    trimestre = (hoy.month - 1) // 3 + 1
    contexto_temporal = f"Fecha de hoy: {hoy.strftime('%d/%m/%Y')} — Trimestre actual: {trimestre}T 2026\n"

    perfil_actual = state.get("perfil", "")
    perfil_linea  = ""
    if perfil_actual:
        label = {"autonomo": "Autónomo", "sociedad": "Sociedad"}.get(perfil_actual, "")
        perfil_linea = f"Perfil del cliente: {label}\n"

    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    messages += [m for m in historial[:-1] if not isinstance(m, RemoveMessage)]
    prompt_con_contexto = (
        f"{contexto_temporal}{perfil_linea}"
        f"Contexto recuperado de la base de conocimiento:\n---\n{contexto}\n---\n\n"
        f"Pregunta del usuario: {historial[-1].content}"
    )
    messages.append(HumanMessage(content=prompt_con_contexto))
    respuesta = _invoke_con_retry(_llms_globales["agente"], messages, tipo="agente")
    return {"messages": [AIMessage(content=respuesta.content)]}

# ── Construcción del grafo ───────────────────────────────────────────────────
workflow = StateGraph(AgentState)
workflow.add_node("podar_historial",      podar_historial)
workflow.add_node("detectar_perfil",      detectar_perfil)
workflow.add_node("clasificar_consulta",  clasificar_consulta)
workflow.add_node("recuperar_plazos",     recuperar_plazos)
workflow.add_node("recuperar_documentos", recuperar_documentos)
workflow.add_node("recuperar_general",    recuperar_general)
workflow.add_node("generar_respuesta",    generar_respuesta)

workflow.add_edge(START,                  "podar_historial")
workflow.add_edge("podar_historial",      "detectar_perfil")
workflow.add_edge("detectar_perfil",      "clasificar_consulta")
workflow.add_conditional_edges("clasificar_consulta", router, {
    "recuperar_plazos":     "recuperar_plazos",
    "recuperar_documentos": "recuperar_documentos",
    "recuperar_general":    "recuperar_general",
})
workflow.add_edge("recuperar_plazos",     "generar_respuesta")
workflow.add_edge("recuperar_documentos", "generar_respuesta")
workflow.add_edge("recuperar_general",    "generar_respuesta")
workflow.add_edge("generar_respuesta",    END)

memory = MemorySaver()
agente = workflow.compile(checkpointer=memory)

print(f"Agente compilado — modelo agente: {MODELOS_AGENTE[_estado['agente']['modelo_idx']]} / clave {_estado['agente']['clave_idx']+1}")
print(f"                — modelo lite:   {MODELOS_LITE[_estado['lite']['modelo_idx']]} / clave {_estado['lite']['clave_idx']+1}")
print("  Grafo: START → podar_historial → detectar_perfil → clasificar_consulta → [recuperar] → generar_respuesta → END")

[WARN] [agente] gemini-3-flash-preview clave 1 agotada (límite diario)...
[WARN] [agente] gemini-3-flash-preview clave 2 agotada (límite diario)...
[INFO] [agente] Arrancando con gemini-3-flash-preview / clave 3
Agente compilado — modelo agente: gemini-3-flash-preview / clave 3
                — modelo lite:   gemini-3.1-flash-lite-preview / clave 1
  Grafo: START → podar_historial → detectar_perfil → clasificar_consulta → [recuperar] → generar_respuesta → END


## 5. Lógica de avisos por antelación

In [20]:
from datetime import date, timedelta

def obtener_obligaciones_proximas(perfil: str, dias_horizonte: int = 60) -> str:
    """Devuelve las obligaciones fiscales próximas en los próximos N días."""
    df = pd.read_csv("../data/calendario_fiscal.csv", sep=",")
    hoy = date.today()
    limite = hoy + timedelta(days=dias_horizonte)

    if perfil in ("autonomo", "sociedad"):
        df = df[df["perfil"].isin([perfil, "ambos"])]

    df["fecha_limite_2026"] = pd.to_datetime(df["fecha_limite_2026"]).dt.date
    df = df[(df["fecha_limite_2026"] >= hoy) & (df["fecha_limite_2026"] <= limite)]
    df = df.sort_values("fecha_limite_2026")

    if df.empty:
        return f"No hay obligaciones fiscales en los próximos {dias_horizonte} días."

    lineas = [f"Obligaciones próximas ({hoy} → {limite}):\n"]
    for _, row in df.iterrows():
        fecha = row["fecha_limite_2026"]
        inicio = fecha - timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"• Modelo {row['modelo']} — {row['nombre']}\n"
            f"  Plazo: {fecha} | Inicio recomendado: {inicio}\n"
        )
    return "\n".join(lineas)

# Ejemplo
print(obtener_obligaciones_proximas("autonomo", dias_horizonte=90))

Obligaciones próximas (2026-05-08 → 2026-08-06):

• Modelo 100 — IRPF anual 2025 — fin campaña
  Plazo: 2026-06-30 | Inicio recomendado: 2026-05-31

• Modelo 130 — Pago fraccionado IRPF autónomos 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 303 — Autoliquidación IVA 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 111 — Retenciones e ingresos a cuenta IRPF 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15

• Modelo 115 — Retenciones e ingresos a cuenta — alquileres 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15



## 6. Moderación en cascada

Filtra preguntas fuera del ámbito fiscal **antes** de llegar al agente mediante tres capas progresivas:

1. **Reglas** — keywords fiscales/offtopic, coste cero, instantáneo
2. **ML** — TF-IDF + LogisticRegression, solo actúa si la confianza ≥ 85 %
3. **LLM** — llamada a Gemini únicamente para los casos ambiguos que superan las dos capas anteriores

Este patrón reduce las llamadas innecesarias a la API en ~80 % para preguntas claramente fuera de ámbito.

In [21]:
import re
import numpy as np
from dataclasses import dataclass
from typing import Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

UMBRAL_CONFIANZA_ML = 0.85

KEYWORDS_FISCAL = re.compile(
    r"\b(modelo|irpf|iva|impuesto|declaraci[oó]n|renta|hacienda|aeat|tribut|fiscal|"
    r"autonomo|aut[oó]nomo|sociedad|empresa|s\.l|factura|casilla|plazo|trimestre|"
    r"303|130|111|115|100|200|202|347|390|retenci[oó]n|deducci[oó]n)\b",
    re.IGNORECASE,
)
KEYWORDS_OFFTOPIC = re.compile(
    r"\b(receta|cocina|deporte|f[uú]tbol|pel[ií]cula|m[uú]sica|viaje|hotel|"
    r"tiempo|clima|meteorolog[ií]a|amor|relaci[oó]n|juego|videojuego)\b",
    re.IGNORECASE,
)

@dataclass
class ResultadoModeracion:
    decision:  str    # "fiscal" | "offtopic"
    capa:      str    # "reglas" | "ml" | "llm"
    confianza: float

_ejemplos = [
    ("¿Cuándo presento el modelo 303?",                        "fiscal"),
    ("¿Qué obligaciones tengo como autónomo?",                 "fiscal"),
    ("¿Cómo se rellena la casilla 01 del IVA?",               "fiscal"),
    ("Plazo para presentar el IRPF 2025",                      "fiscal"),
    ("¿Qué es la domiciliación en el modelo 130?",             "fiscal"),
    ("Retenciones en el modelo 111 del segundo trimestre",     "fiscal"),
    ("¿Cuánto tiempo tengo para presentar el IS?",             "fiscal"),
    ("Deducciones en el modelo 303",                           "fiscal"),
    ("¿Cómo me doy de alta como autónomo en hacienda?",        "fiscal"),
    ("¿Qué es el pago fraccionado del IRPF?",                  "fiscal"),
    ("¿Cuál es la mejor receta de paella?",                    "offtopic"),
    ("¿Quién ganó el partido de ayer?",                        "offtopic"),
    ("Recomiéndame una película de terror",                    "offtopic"),
    ("¿Qué tiempo hace en Madrid?",                            "offtopic"),
    ("¿Cómo se llama el presidente de Francia?",               "offtopic"),
    ("Cuéntame un chiste",                                     "offtopic"),
    ("¿Cuál es la capital de Australia?",                      "offtopic"),
    ("Dame una ruta de senderismo",                            "offtopic"),
]
_X, _y = zip(*_ejemplos)
clasificador_ml = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf",   LogisticRegression(max_iter=1000, random_state=42)),
])
clasificador_ml.fit(list(_X), list(_y))

def _extraer_texto(msg) -> str:
    """Extrae el texto de un AIMessage cuyo .content puede ser str o list."""
    c = msg.content
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        partes = [p.get("text", "") if isinstance(p, dict) else str(p) for p in c]
        return " ".join(partes)
    return str(c)

def moderar_pregunta(texto: str) -> ResultadoModeracion:
    """Cascada: Reglas → ML → LLM-lite para clasificar si la pregunta es fiscal."""
    if KEYWORDS_FISCAL.search(texto):
        return ResultadoModeracion("fiscal",   "reglas", 1.0)
    if KEYWORDS_OFFTOPIC.search(texto):
        return ResultadoModeracion("offtopic", "reglas", 1.0)

    proba     = clasificador_ml.predict_proba([texto])[0]
    clases    = clasificador_ml.classes_
    idx_max   = int(np.argmax(proba))
    confianza = float(proba[idx_max])
    if confianza >= UMBRAL_CONFIANZA_ML:
        return ResultadoModeracion(clases[idx_max], "ml", confianza)

    # Capa 3: modelo lite con fallback completo
    prompt = (
        "Clasifica esta pregunta como 'fiscal' o 'offtopic'. "
        "Responde SOLO con una palabra.\n\nPregunta: " + texto
    )
    try:
        raw = _extraer_texto(
            _invoke_con_retry(_llms_globales["lite"], [HumanMessage(content=prompt)], tipo="lite")
        ).strip().lower()
        decision = "fiscal" if "fiscal" in raw else "offtopic"
        return ResultadoModeracion(decision, "llm", 0.6)
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
            print(f"[WARN] Cuota agotada en moderación — asumiendo 'fiscal' para: {texto[:60]}")
            return ResultadoModeracion("fiscal", "llm-fallback", 0.5)
        raise

# ── Prueba de la moderación ──────────────────────────────────────────────────
casos_prueba = [
    "¿Cuándo presento el modelo 303 del 2T?",
    "¿Cuál es la mejor receta de tortilla?",
    "Necesito información sobre retenciones IRPF",
    "¿Me recomiendas una serie en Netflix?",
    "¿Puedo deducir el alquiler de mi oficina?",
]

print("Prueba de moderación en cascada:\n")
for texto in casos_prueba:
    resultado = moderar_pregunta(texto)
    print(f"  [{resultado.decision.upper():8s}] [{resultado.capa:12s}] {texto}")

Prueba de moderación en cascada:

  [FISCAL  ] [reglas      ] ¿Cuándo presento el modelo 303 del 2T?
  [OFFTOPIC] [reglas      ] ¿Cuál es la mejor receta de tortilla?
  [FISCAL  ] [reglas      ] Necesito información sobre retenciones IRPF
  [OFFTOPIC] [llm         ] ¿Me recomiendas una serie en Netflix?
  [FISCAL  ] [llm         ] ¿Puedo deducir el alquiler de mi oficina?


## 7. Herramientas fiscales con @tool y loop ReAct

**Demo alternativa al agente principal.** Mientras el agente de las secciones 4–6 usa un pipeline RAG explícito con routing condicional, este agente ReAct delega la decisión de qué herramienta usar al propio LLM en cada turno. El patrón crea un ciclo `agente → tools → agente` que se repite hasta que el LLM produce una respuesta final sin `tool_calls`.

**Cuándo usar cada arquitectura:**
- **Agente principal (sección 4)** — control total del flujo, routing determinista, ideal para producción.
- **Agente ReAct (esta sección)** — el LLM decide dinámicamente qué herramienta invocar; más flexible pero menos predecible.

Herramientas definidas:
- `consultar_obligaciones_proximas` — obligaciones en los próximos N días para un perfil
- `consultar_plazo_modelo` — plazo exacto de un modelo fiscal concreto

> **Nota:** este agente combina las tools del calendario con contexto RAG de los manuales para que las respuestas sean completas tanto en plazos como en procedimiento.

In [22]:
import datetime
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

_df_cal = pd.read_csv("../data/calendario_fiscal.csv")
_df_cal["fecha_limite_2026"] = pd.to_datetime(_df_cal["fecha_limite_2026"]).dt.date

@tool
def consultar_obligaciones_proximas(perfil: str, dias_horizonte: int = 60) -> str:
    """Devuelve las obligaciones fiscales próximas para un perfil dado.
    Usar cuando el usuario pregunte qué tiene pendiente en los próximos días o meses.
    perfil: 'autonomo', 'sociedad' o 'ambos'
    dias_horizonte: número de días a mirar hacia adelante (por defecto 60)
    """
    hoy   = datetime.date.today()
    hasta = hoy + datetime.timedelta(days=dias_horizonte)
    df = _df_cal.copy()
    if perfil in ("autonomo", "sociedad"):
        df = df[df["perfil"].isin([perfil, "ambos"])]
    df = df[(df["fecha_limite_2026"] >= hoy) & (df["fecha_limite_2026"] <= hasta)]
    df = df.sort_values("fecha_limite_2026")
    if df.empty:
        return f"No hay obligaciones en los próximos {dias_horizonte} días."
    lineas = [f"Obligaciones {hoy} → {hasta}:\n"]
    for _, row in df.iterrows():
        inicio = row["fecha_limite_2026"] - datetime.timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"• Modelo {row['modelo']} — {row['nombre']}\n"
            f"  Plazo: {row['fecha_limite_2026']} | Inicio recomendado: {inicio}"
        )
    return "\n".join(lineas)

@tool
def consultar_plazo_modelo(modelo: str) -> str:
    """Devuelve el plazo exacto de un modelo fiscal concreto.
    Usar cuando el usuario pregunte por la fecha límite de un modelo específico.
    modelo: número del modelo (ej: '303', '130', '111')
    """
    df = _df_cal[_df_cal["modelo"].astype(str) == str(modelo)]
    if df.empty:
        return f"No encontré información del modelo {modelo} en el calendario."
    lineas = []
    for _, row in df.iterrows():
        inicio = row["fecha_limite_2026"] - datetime.timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"Modelo {row['modelo']} — {row['nombre']} ({row['perfil']})\n"
            f"  Plazo: {row['fecha_limite_2026']} | Inicio recomendado: {inicio}"
        )
    return "\n".join(lineas)

tools_fiscales = [consultar_obligaciones_proximas, consultar_plazo_modelo]
nodo_tools     = ToolNode(tools_fiscales)
llm_con_tools  = _llms_globales["agente"].bind_tools(tools_fiscales)

# ── Grafo ReAct con tools ────────────────────────────────────────────────────
from typing import Literal as _Literal

class AgentStateReAct(TypedDict):
    messages:     Annotated[list, operator.add]
    perfil:       str
    contexto_rag: str

def _recuperar_contexto_react(pregunta: str, perfil: str) -> str:
    """Recupera contexto RAG de manuales y calendario para enriquecer la respuesta ReAct."""
    search_kwargs = {"k": 6}
    if perfil in ("autonomo", "sociedad"):
        search_kwargs["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_kwargs).invoke(pregunta)
    docs_csv = vectorstore.as_retriever(
        search_kwargs={"k": 4, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}}
    ).invoke(pregunta)
    return _combinar_docs(docs_manuales_res, docs_csv)

def nodo_agente_react(state: AgentStateReAct) -> AgentStateReAct:
    # Recupera contexto RAG en cada turno para que el LLM tenga información
    # de los manuales además de lo que devuelvan las tools del calendario.
    ultima_pregunta = next(
        (m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), ""
    )
    contexto = _recuperar_contexto_react(ultima_pregunta, state.get("perfil", ""))

    fecha_hoy = datetime.date.today().strftime("%d/%m/%Y")
    perfil_linea = ""
    if state.get("perfil"):
        label = {"autonomo": "Autónomo", "sociedad": "Sociedad"}.get(state["perfil"], "")
        perfil_linea = f"Perfil del cliente: {label}\n"
    system = (
        SYSTEM_PROMPT
        + f"\n\nFecha de hoy: {fecha_hoy}\n{perfil_linea}"
        + f"\nContexto RAG (manuales y calendario):\n---\n{contexto}\n---"
    )
    messages = [SystemMessage(content=system)] + [m for m in state["messages"] if not isinstance(m, RemoveMessage)]
    respuesta = _invoke_con_retry(llm_con_tools, messages, tipo="agente")
    return {"messages": [respuesta]}

def router_react(state: AgentStateReAct) -> _Literal["tools", "__end__"]:
    ultimo = state["messages"][-1]
    if hasattr(ultimo, "tool_calls") and ultimo.tool_calls:
        return "tools"
    return "__end__"

workflow_react = StateGraph(AgentStateReAct)
workflow_react.add_node("agente", nodo_agente_react)
workflow_react.add_node("tools",  nodo_tools)
workflow_react.add_edge(START, "agente")
workflow_react.add_conditional_edges("agente", router_react, {"tools": "tools", "__end__": END})
workflow_react.add_edge("tools", "agente")

agente_react = workflow_react.compile(checkpointer=MemorySaver())

# ── Prueba del agente ReAct ──────────────────────────────────────────────────
import uuid as _uuid
_cfg_react = {"configurable": {"thread_id": str(_uuid.uuid4())}}

preguntas_react = [
    "Soy autónomo, ¿qué tengo pendiente en los próximos 60 días?",
    "¿Cuál es el plazo del modelo 303?",
]
_state_react = {"messages": [], "perfil": "autonomo", "contexto_rag": ""}

for p in preguntas_react:
    _state_react["messages"] = _state_react.get("messages", []) + [HumanMessage(content=p)]
    _result = agente_react.invoke(_state_react, config=_cfg_react)
    _state_react = _result
    respuesta_react = next(
        (m.content for m in reversed(_result["messages"]) if isinstance(m, AIMessage) and m.content), ""
    )
    print(f"\nPregunta: {p}")
    print(f"Respuesta: {respuesta_react[:400]}...")

[RAG] Documentos recuperados: 10 (manual_iva_303_2025.pdf, manual_renta_100_130_2025_parte1.pdf, manual_renta_100_130_2025_parte2.pdf, obligaciones_perfil.csv, calendario_fiscal.csv)
[RAG] Documentos recuperados: 10 (manual_iva_303_2025.pdf, manual_renta_100_130_2025_parte1.pdf, manual_renta_100_130_2025_parte2.pdf, obligaciones_perfil.csv, calendario_fiscal.csv)
[RAG] Documentos recuperados: 10 (manual_iva_303_2025.pdf, manual_renta_100_130_2025_parte1.pdf, manual_renta_100_130_2025_parte2.pdf, obligaciones_perfil.csv, calendario_fiscal.csv)

Pregunta: Soy autónomo, ¿qué tengo pendiente en los próximos 60 días?
Respuesta: [{'type': 'text', 'text': '**Perfil:** Autónomo.\n\nDe acuerdo con la fecha actual (8 de mayo de 2026) y el horizonte de 60 días solicitado, estas son tus obligaciones fiscales pendientes, incluyendo el cierre del segundo trimestre (2T):\n\n### Obligación Inmediata (Campaña de Renta)\n**Modelo 100 — Impuesto sobre la Renta de las Personas Físicas (Anual 2025):**\n*  

## 8. LLM-as-Judge — evaluación de calidad

Un segundo LLM evalúa cada respuesta del agente con puntuación estructurada en tres dimensiones: **precisión técnica**, **claridad** y **completitud**. Permite detectar sistemáticamente cuándo el agente responde mal sin revisar manualmente cada caso.

In [23]:
import json
import uuid
import time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

PROMPT_JUEZ = ChatPromptTemplate.from_template("""
Eres un evaluador experto en asesoría fiscal española.
Evalúa la calidad de esta respuesta según los criterios dados.

**Pregunta:** {pregunta}
**Respuesta a evaluar:** {respuesta}
**Criterios:** {criterios}

Responde ÚNICAMENTE con este JSON (sin markdown ni bloques de código):
{{
  "puntuacion_global": número entre 1 y 10,
  "precision_tecnica": número entre 1 y 10,
  "claridad": número entre 1 y 10,
  "completitud": número entre 1 y 10,
  "justificacion": "máximo 2 oraciones"
}}
""")

def evaluar_respuesta(pregunta: str, respuesta: str) -> dict:
    """Evalúa una respuesta del agente con LLM-as-Judge usando modelo lite."""
    criterios = "precisión de fechas y plazos, inclusión de domiciliación bancaria cuando aplique, claridad de la explicación, completitud según el perfil del usuario"
    prompt_messages = PROMPT_JUEZ.format_messages(
        pregunta=pregunta, respuesta=respuesta, criterios=criterios
    )
    raw = _extraer_texto(
        _invoke_con_retry(_llms_globales["lite"], prompt_messages, tipo="lite")
    ).strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

def evaluar_pipeline(pares_pregunta_respuesta: list[tuple[str, str]], nombre: str = "Agente") -> dict:
    """Evalúa un conjunto de pares pregunta/respuesta y devuelve métricas agregadas."""
    evaluaciones = []
    for pregunta, respuesta in pares_pregunta_respuesta:
        ev = evaluar_respuesta(pregunta, respuesta)
        ev["pregunta"]   = pregunta
        ev["respuesta"]  = respuesta
        evaluaciones.append(ev)
        print(f"  [{ev['puntuacion_global']}/10] {pregunta[:70]}")

    scores = [e["puntuacion_global"] for e in evaluaciones]
    return {
        "nombre":         nombre,
        "score_promedio": round(float(np.mean(scores)), 2),
        "score_min":      min(scores),
        "score_max":      max(scores),
        "evaluaciones":   evaluaciones,
    }

# ── Benchmark con respuestas REALES del agente ───────────────────────────────
preguntas_benchmark = [
    ("Soy autónomo, ¿cuándo presento el modelo 303 del 2T?",        "autonomo"),
    ("¿Qué obligaciones tiene una sociedad en el segundo trimestre?", "sociedad"),
    ("¿Cómo se calcula la casilla 01 del modelo 303?",               "autonomo"),
]

print("Generando respuestas reales del agente...\n")
pares_reales = []
for pregunta, perfil in preguntas_benchmark:
    cfg   = {"configurable": {"thread_id": str(uuid.uuid4())}}
    state = {
        "messages":      [HumanMessage(content=pregunta)],
        "perfil":        perfil,
        "contexto_rag":  "",
        "tipo_consulta": "",
    }
    result         = agente.invoke(state, config=cfg)
    ultimo_msg     = result["messages"][-1]
    respuesta_real = _extraer_texto(ultimo_msg)
    pares_reales.append((pregunta, respuesta_real))
    print(f"✓ Respuesta generada para: {pregunta[:60]}...")

print("\nEvaluando con LLM-as-Judge (modelo lite)...\n")
resultados = evaluar_pipeline(pares_reales, nombre="Agente Fiscal v2")

print(f"\nScore promedio: {resultados['score_promedio']}/10")
print(f"Rango: {resultados['score_min']} – {resultados['score_max']}")
print("\nDetalle por pregunta:")
for ev in resultados["evaluaciones"]:
    print(f"\n  Pregunta:    {ev['pregunta'][:80]}")
    print(f"  Global: {ev['puntuacion_global']} | Precisión: {ev['precision_tecnica']} | Claridad: {ev['claridad']} | Completitud: {ev['completitud']}")
    print(f"  {ev['justificacion']}")
    print(f"\n  Respuesta del agente:\n{'-'*40}")
    print(ev["respuesta"])

Generando respuestas reales del agente...

[Router] Consulta clasificada como: 'plazos'
[RAG] Documentos recuperados: 10 (calendario_fiscal.csv, obligaciones_perfil.csv)
✓ Respuesta generada para: Soy autónomo, ¿cuándo presento el modelo 303 del 2T?...
[Router] Consulta clasificada como: 'plazos'
[RAG] Documentos recuperados: 13 (calendario_fiscal.csv, manual_sociedades_200_202_2024.pdf, obligaciones_perfil.csv)
✓ Respuesta generada para: ¿Qué obligaciones tiene una sociedad en el segundo trimestre...
[Router] Consulta clasificada como: 'documentos'
[RAG] Documentos recuperados: 12 (manual_iva_303_2025.pdf, manual_renta_100_130_2025_parte1.pdf, calendario_fiscal.csv)
✓ Respuesta generada para: ¿Cómo se calcula la casilla 01 del modelo 303?...

Evaluando con LLM-as-Judge (modelo lite)...

  [9/10] Soy autónomo, ¿cuándo presento el modelo 303 del 2T?
  [8/10] ¿Qué obligaciones tiene una sociedad en el segundo trimestre?
  [3/10] ¿Cómo se calcula la casilla 01 del modelo 303?

Score prome

## 9. Demo interactiva

Ejecuta la celda siguiente para abrir un chat con el agente. Escribe `salir` para terminar la sesión.

In [ ]:
import uuid

def chat(perfil_inicial: str = ""):
    """Sesión de chat interactiva con el agente fiscal."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    state = {"messages": [], "perfil": perfil_inicial, "contexto_rag": "", "tipo_consulta": ""}

    print("=" * 60)
    print("  ASISTENTE FISCAL — Gestoría España")
    print("=" * 60)
    if perfil_inicial:
        print(f"  Perfil activo: {perfil_inicial.upper()}")
    print("  Escribe 'salir' para terminar.\n")

    while True:
        pregunta = input("Tú: ").strip()
        if pregunta.lower() in ("salir", "exit", "quit"):
            print("Sesión finalizada.")
            break
        if not pregunta:
            continue

        state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
        result = agente.invoke(state, config=config)
        state = result

        respuesta = result["messages"][-1].content
        print(f"\nAsistente: {respuesta}\n")
        print("-" * 60)

# Ejecuta el chat solo si hay stdin interactivo real.
# nbconvert no soporta input() — capturamos la excepción para no romper la ejecución automática.
try:
    chat(perfil_inicial="")
except EOFError:
    print("Chat omitido en ejecución automática — ejecuta esta celda manualmente en Jupyter.")
except Exception as e:
    if "raw_input was called" in str(e) or "StdinNotImplemented" in type(e).__name__:
        print("Chat omitido en ejecución automática — ejecuta esta celda manualmente en Jupyter.")
    else:
        raise

  ASISTENTE FISCAL — Gestoría España
  Escribe 'salir' para terminar.



Sesión finalizada.


---

### Casos de prueba documentados

Los 5 casos mínimos requeridos se prueban en las celdas siguientes (sin entrada interactiva).

In [25]:
def preguntar(pregunta: str, state: dict, config: dict) -> tuple[str, dict]:
    state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
    result = agente.invoke(state, config=config)
    return result["messages"][-1].content, result

# Sesión de prueba
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}
state = {"messages": [], "perfil": "", "contexto_rag": "", "tipo_consulta": ""}

casos = [
    # Caso 1 — Plazo de un modelo concreto → router: plazos
    "¿Cuál es el plazo para presentar el modelo 303 del primer trimestre de 2026?",
    # Caso 2 — Cómo rellenar una casilla → router: documentos
    "¿Cómo se calcula la casilla 01 del modelo 303?",
    # Caso 3 — Obligaciones autónomo 1T → router: plazos
    "Soy autónomo en estimación directa. ¿Qué declaraciones tengo que presentar en el primer trimestre?",
    # Caso 4 — Obligaciones sociedad 2T → router: plazos
    "Somos una sociedad limitada. ¿Qué obligaciones fiscales tenemos en el segundo trimestre?",
    # Caso 5 — Pregunta encadenada (memoria + antelación) → router: plazos
    "¿Y cuándo debería empezar a preparar esas declaraciones para llegar a tiempo?",
]

for i, pregunta in enumerate(casos, 1):
    print(f"\n{'='*60}")
    print(f"CASO {i}: {pregunta}")
    print('='*60)
    respuesta, state = preguntar(pregunta, state, config)
    print(f"RESPUESTA:\n{respuesta}")


CASO 1: ¿Cuál es el plazo para presentar el modelo 303 del primer trimestre de 2026?
[Router] Consulta clasificada como: 'plazos'
[RAG] Documentos recuperados: 10 (calendario_fiscal.csv)
RESPUESTA:
[{'type': 'text', 'text': 'Para darte la información correcta y conocer el resto de tus obligaciones fiscales, necesito saber tu perfil. ¿Eres autónomo o representas a una sociedad?\n\nNo obstante, para el **Modelo 303 (IVA)**, el plazo es el mismo para ambos perfiles. Ten en cuenta que, a fecha de hoy (8 de mayo de 2026), el plazo para el primer trimestre ya ha finalizado:\n\n**Modelo 303 — Autoliquidación IVA 1T 2026:**\n- **Fecha límite:** 20 de abril de 2026 (Plazo vencido)\n- **Domiciliación hasta:** 15 de abril de 2026\n- **Inicio de preparación recomendado:** 10 de abril de 2026 (10 días antes)\n\n*Fuente: Calendario fiscal AEAT 2026*', 'extras': {'signature': '...'}}]

CASO 2: ¿Cómo se calcula la casilla 01 del modelo 303?
[Router] Consulta clasificada como: 'documentos'
[RAG] Docum

---

## 10. Demo de presentación

Dos escenarios reales que cubren los casos de uso principales del agente:

- **Escenario A — Autónomo, primer trimestre:** consulta de obligaciones, plazos, cómo rellenar una casilla y pregunta encadenada con memoria.
- **Escenario B — Sociedad, cierre de ejercicio:** obligaciones del cuarto trimestre + cierre anual, pagos fraccionados, antelación y pregunta encadenada.


In [26]:
import uuid

def demo_escenario(titulo: str, casos: list) -> None:
    """Ejecuta una secuencia de preguntas en la misma sesión (mismo thread) para demostrar memoria."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    state = {"messages": [], "perfil": "", "contexto_rag": "", "tipo_consulta": ""}

    separador = "#" * 70
    print(f"{separador}")
    print(f"  {titulo}")
    print(f"{separador}")

    for i, pregunta in enumerate(casos, 1):
        print(f"--- Pregunta {i} ---")
        print(f"Usuario: {pregunta}")
        
        state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
        result = agente.invoke(state, config=config)
        state = result
        respuesta = result["messages"][-1].content
        print(f"Agente: {respuesta}")
        print()


# Escenario A: Autonomo, primer trimestre
# Demuestra: obligaciones 1T, plazo concreto, cumplimentacion y pregunta encadenada con memoria.
casos_autonomo_1t = [
    "Soy autonomo en estimacion directa. Que declaraciones tengo que presentar en el primer trimestre de 2026?",
    "Cual es exactamente la fecha limite del modelo 130 y cuando deberia empezar a prepararlo?",
    "Como calculo el importe a ingresar en el modelo 130? Que pongo en la casilla 03?",
    "Y si tengo retenciones de clientes que me han practicado? Como las resto?",
]

demo_escenario("ESCENARIO A: Autonomo — obligaciones 1T 2026", casos_autonomo_1t)


# Escenario B: Sociedad, cierre de ejercicio
# Demuestra: obligaciones 4T + cierre anual, IS, pago fraccionado y pregunta encadenada.
casos_sociedad_cierre = [
    "Somos una S.L. Que obligaciones fiscales tenemos en el cuarto trimestre de 2026 y a principios de 2027?",
    "Cuando tenemos que presentar el Impuesto de Sociedades (modelo 200) y con cuanta antelacion hay que prepararlo?",
    "Tenemos que hacer algun pago fraccionado del IS en octubre? Que es el modelo 202?",
    "Como se calcula la base del pago fraccionado del modelo 202?",
]

demo_escenario("ESCENARIO B: Sociedad — cierre de ejercicio 2026", casos_sociedad_cierre)


######################################################################
  ESCENARIO A: Autonomo — obligaciones 1T 2026
######################################################################
--- Pregunta 1 ---
Usuario: Soy autonomo en estimacion directa. Que declaraciones tengo que presentar en el primer trimestre de 2026?
[Perfil] Detectado: autónomo
[Router] Consulta clasificada como: 'plazos'
[RAG] Documentos recuperados: 13 (calendario_fiscal.csv, manual_renta_100_130_2025_parte1.pdf, obligaciones_perfil.csv)
Agente: [{'type': 'text', 'text': '**Perfil:** Autónomo (Estimación directa).\n\nTen en cuenta que, a fecha de hoy (8 de mayo de 2026), los plazos para las declaraciones trimestrales del primer trimestre (1T) ya han vencido. Si no las has presentado, estarían fuera de plazo.\n\nEstas son las obligaciones que correspondían al primer trimestre de 2026:\n\n**Modelo 303 — IVA 1T 2026:**\n*   **Fecha límite:** 20 de abril de 2026\n*   **Domiciliación hasta:** 15 de abril de 2026\n*  